# Zürich Tram Flow
### Verspätungsanalyse und Vorhersage im Tramnetz Zürich

> **Datenbasis:** [`sf_data-research`](https://github.com/kaywiegand/sf_data-research) — Research & Data Engineering Phase (abgeschlossen)  
> **Erstellt mit:** [wgnd-scaffolding](https://github.com/kaywiegand/wgnd-scaffolding) · [wgnd-toolkit](https://github.com/kaywiegand/wgnd-toolkit)

## Project Facts

| Feld | Wert |
|------|------|
| **Business-Frage** | Wo, wann und warum entstehen Verspätungen im Zürcher Tramnetz — und lassen sie sich vorhersagen? |
| **Stakeholder** | VBZ (Betreiber), Stadtplanung Zürich, Fahrgäste |
| **Methode** | EDA → Korrelationsanalyse → Zeitreihen-/ML-Modell → Dashboard |
| **Hauptdatenquelle** | VBZ IST-Daten 2023–2025 (opentransportdata.swiss) + GTFS + Meteo + Events |
| **Ziel-Metrik** | Vorhersagegenauigkeit (MAE) pro Linie/Stadtkreis; On-Time Performance (OTP) |
| **Out of Scope** | Echtzeit-Feed (GTFS-RT), Daten ab Format v2 (ab Mitte 2025), VBB Berlin |
| **Analysezeitraum** | 2023–2025 (IST-Daten Format v1, einheitlich) |
| **Stack** | Python · Polars · Pandas · GeoPandas · Plotly · Folium |

## Project Context

### Scenario

Verspätungen im öffentlichen Nahverkehr sind ärgerlich — für Menschen und für das System.
Das Zürcher Tramnetz (VBZ) bietet eine außergewöhnlich gute Open-Data-Grundlage:
IST-Daten mit Echtzeit-Verspätungen pro Haltestelle, GTFS-Fahrplandaten, Wetterdaten
und Eventkalender — über drei Jahre.

Das Tram fährt im offenen Stadtverkehr — beeinflusst durch Autos, Fußgänger, Wetter,
Topografie und Großveranstaltungen. Das macht es zu einem besonders interessanten
Analysegegenstand für Betreiber, Stadtplanung und Fahrgäste.

### Mission

Aufbau einer vollständigen Analyse- und Vorhersage-Pipeline für Verspätungen im
Zürcher Tramnetz — vom validierten Master-Datensatz bis zum interaktiven Dashboard.
Zürich dient dabei als Referenzmodell für Städte wie Berlin, die ihre Datenpotenziale
noch nicht ausschöpfen.

### Zentrale Fragen

* Wo entstehen Verspätungen im Tramnetz — und zu welchen Zeiten?
* Welche Einflussfaktoren spielen die größte Rolle? (Wetter, Tageszeit, Events, Topografie)
* Lassen sich Verspätungen vorhersagen, bevor sie entstehen?
* Welche Haltestellen oder Streckenabschnitte lösen Kettenreaktionen aus?
* Was kann ein Betreiber oder eine Stadt konkret besser machen?

### Methode & Metriken

| Metrik | Zielwert | Begründung |
|--------|----------|-------------|
| On-Time Performance (OTP) | Baseline messen | Anteil Fahrten < 2 Min Verspätung |
| Mean Absolute Error (MAE) | < 60 Sek | Vorhersagegenauigkeit pro Linie/Stadtkreis |
| Bottleneck Score | Top-10 Haltestellen | Haltestellen mit systemweiten Folgeverspätungen |
| Weather Sensitivity Score | Korrelation quantifizieren | Einfluss von Regen/Wind/Temperatur |
| Event Impact Score | Verspätungsanstieg messbar | Vergleich Event-Tage vs. normale Tage |

---

## Datenbasis & Data Engineering

Die gesamte Data-Engineering-Phase wurde in einem separaten Research-Repo durchgeführt
und ist dort vollständig dokumentiert:

> **Quelle:** [`sf_data-research`](https://github.com/kaywiegand/sf_data-research)  
> **Status:** Phase 1 abgeschlossen — Datenbasis vollständig und validiert.

### Was wurde dort gemacht?

| Schritt | Beschreibung | Notebook |
| :--- | :--- | :--- |
| IST-Daten | Download 36 ZIP-Archive (38 GB), Filter auf VBZ & Tram, Parquet-Konvertierung | `vbz-ist-daten.ipynb` |
| GTFS | Fahrplandaten 2023–2025, Spatial Join Stadtkreise, Haltestellen-Lookup | `vbz-gtfs-data.ipynb` |
| Meteo | 3 Quellen konsolidiert (Stampfenbachstr. + Mythenquai), Stundenmittelwerte | `vbz-meteo-data.ipynb` |
| Events | 301 Einträge, 5 Kategorien, Gewichtungsschema 1–3 | `vbz-events-data.ipynb` |
| Benchmark | Polars vs. Pandas: 4× schneller, 4× weniger RAM | `vbz-pandas-vs-polars.ipynb` |
| Master-Merge | Left Join IST + GTFS + Meteo + Events → `vbz_master.parquet` | `vbz-data-master-preparation.ipynb` |
| Validierung | 8 Checks: Schema, Abdeckung, Wertebereiche, Nulls, Join-Qualität, Business-Logik | `vbz-data-master-validation.ipynb` |

### Wichtige Entscheidungen aus der Research-Phase

| Entscheidung | Was | Warum |
| :--- | :--- | :--- |
| Polars statt Pandas | Haupt-DataFrame-Bibliothek | 4× schneller, 4× weniger RAM bei 94 Mio. Zeilen |
| Left Join überall | Merge-Strategie | Kein Datenverlust durch Join-Lücken |
| 2024 als GTFS-Referenzjahr | Fahrplandaten | Vollständigste Datenlage, stabilstes Jahr |
| 2 Meteo-Stationen | Stampfenbachstrasse + Mythenquai | Zwei Topografien: Stadtlage vs. Seelage |
| Scope 2023–2025 v1 | Analysezeitraum | Einheitliches Datenformat, kein Mischformat |
| Stadtkreis im Lookup | district im GTFS-Join | Einmalig sauber im Master, kein wiederholter Spatial Join |
| Ausfälle behalten | `canceled = True` | Extremster Verspätungsfall, für Modell unverzichtbar |
| Schwellenwert Events | >1.000 Besucher | Kleinere Events kein messbarer Netzeinfluss |

### Datenmenge

| Stufe | Menge |
| :--- | :--- |
| Rohdaten (schweizweit, komprimiert) | ~38 GB (36 ZIP-Archive) |
| Rohdaten entpackt | ~500–720 GB |
| Nach Filter VBZ + Tram (Parquet) | ~1,44 GB (1.096 Dateien) |
| Master-Datensatz | ~486 MB · 94 Mio. Zeilen · 24 Spalten |

---

## Data Dictionary — Master-Datensatz

**Datei:** `data/raw/zh-tram-data-master.parquet`  
**Zeilen:** ~88 Millionen · **Spalten:** 24 · **Zeitraum:** 2023–2025

### IST-Daten (Verkehr)

| # | Spaltenname | Typ | Beschreibung |
| :--- | :--- | :--- | :--- |
| 1 | `operating_date` | `Date` | Betriebstag |
| 2 | `line_name` | `Categorical` | Tramliniennummer (z.B. `"11"`) |
| 3 | `bpuic` | `Int32` | Haltestellen-ID — Join-Schlüssel zu GTFS |
| 4 | `arrival_schedule` | `Datetime` | Planmäßige Ankunftszeit |
| 5 | `arrival_delay` | `Float32` | Verspätung Ankunft in **Sekunden** (negativ = zu früh) |
| 6 | `departure_schedule` | `Datetime` | Planmäßige Abfahrtszeit |
| 7 | `departure_delay` | `Float32` | Verspätung Abfahrt in **Sekunden** |
| 8 | `canceled` | `Boolean` | Ausfall = `True` — bewusst behalten als Extremfall |

### GTFS (Fahrplan & Geodaten)

| # | Spaltenname | Typ | Beschreibung |
| :--- | :--- | :--- | :--- |
| 9 | `stop_name` | `Categorical` | Haltestellenname (z.B. `"Paradeplatz"`) |
| 10 | `stop_lat` | `Float32` | Breitengrad (WGS84) |
| 11 | `stop_lon` | `Float32` | Längengrad (WGS84) |
| 12 | `district_nr` | `Int8` | Stadtkreis 1–12 (`null` = außerhalb Stadtgebiet) |
| 13 | `district_name` | `Categorical` | Stadtkreisname (z.B. `"Kreis 1"`) |

### Meteo-Daten (Wetter)

| # | Spaltenname | Typ | Beschreibung |
| :--- | :--- | :--- | :--- |
| 14 | `temperature` | `Float32` | Temperatur in °C |
| 15 | `humidity` | `Float32` | Relative Luftfeuchtigkeit in % |
| 16 | `rain_duration` | `Float32` | Regendauer in min/h |
| 17 | `precipitation` | `Float32` | Niederschlagsmenge in mm |
| 18 | `wind_speed` | `Float32` | Windgeschwindigkeit in km/h |
| 19 | `global_radiation` | `Float32` | Globalstrahlung in W/m² |
| 20 | `flood_intensity` | `Int16` | Überschwemmungsindikator (ERZ-Meldungen) |

### Event-Daten

| # | Spaltenname | Typ | Beschreibung |
| :--- | :--- | :--- | :--- |
| 21 | `event_name` | `Categorical` | Name des Events (`null` = kein Event an diesem Tag) |
| 22 | `event_type` | `Categorical` | Kategorie: `Feiertag`, `Stadtfest`, `Konzert`, `Messe`, `Fussball` |
| 23 | `event_size` | `Int8` | Gewichtung: `1` = mittel (>1k), `2` = groß (10k–30k), `3` = sehr groß (>30k) |
| 24 | `event_location` | `Categorical` | Veranstaltungsort (`null` = kein Event) |

### Join-Strategie

| Join | Schlüssel | Typ |
| :--- | :--- | :--- |
| IST + GTFS Stops | `bpuic` = `bpuic` | Left Join |
| IST + Meteo | `floor(arrival_schedule, '1h')` = `date_time` | Left Join |
| IST + Events | `date(operating_date)` = `Datum` | Left Join |

> **Left Join überall:** Jede Tram-Fahrt bleibt im Datensatz erhalten. Fehlende Werte (z.B. Haltestellen außerhalb Stadtgebiet, Stunden ohne Wetterdaten) erscheinen als `null`.

---

## GTFS-Referenztabellen

**Verzeichnis:** `data/raw/gtfs/` — Referenzjahr 2024 (vollständigste Datenlage)

| Datei | Beschreibung | Verwendung |
| :--- | :--- | :--- |
| `gtfs_stops_lookup.parquet` | Haltestellen-Lookup: `bpuic` → `stop_name`, Koordinaten, `district_nr`, `district_name` | Join-Tabelle im Master |
| `gtfs_tram_stops.parquet` | Alle VBZ-Tram-Haltestellen mit Koordinaten | Geo-Visualisierungen |
| `gtfs_tram_routes.parquet` | Tramlinien (Route-ID, Linienname, Farbe) | Linien-Visualisierungen |
| `gtfs_tram_shapes.parquet` | Tram-Streckenverläufe als Koordinaten-Sequenzen | Streckenkarte |
| `gtfs_tram_trips.parquet` | Fahrten (Trip-ID, Route-ID, Shape-ID) | Verknüpfung Fahrten ↔ Strecken |
| `gtfs_zurich_stops.parquet` | Alle Zürich-Haltestellen (ZVV, nicht nur Tram) | Gesamtnetz-Überblick |
| `gtfs_zurich_routes.parquet` | Alle ZVV-Linien | Gesamtnetz-Überblick |
| `gtfs_zurich_shapes.parquet` | Alle ZVV-Streckenverläufe | Gesamtnetz-Karte |
| `gtfs_zurich_trips.parquet` | Alle ZVV-Fahrten | Gesamtnetz-Überblick |

---

## Workflow

### Phasen

| Phase | 00 Introduction | 01 Exploration | 02 Preparation | 03 Analysis | 04 Insights |
|-------|-----------------|----------------|----------------|-------------|-------------|
| | Projektkontext | Data Profiling | Feature Engineering | Modellierung | Reporting |
| | Data Dictionary | Verteilungen | Cleaning & Transformation | Training & Eval | Visualisierungen |
| | Datenbeschreibung | Erste Hypothesen | Outlier Handling | Vorhersagen | Executive Summary |
| | Quellen & Entscheidungen | Korrelationen | Split-Strategie | Validierung | Dashboard-Vorbereitung |

### Konventionen

```
Mit Polars wird mit primär mit einem Lazy-Frame gearbeitet >> fl
Es wird noch ein oder mehrere Samplesl geben z.B. >> fl_eda und analog df_eda

UPDATEN!

```

| Variable | Phase | Zustand |
|----------|-------|---------|
| `df_raw` | Loading | Originalzustand — schreibgeschützt |
| `df_eda` | Exploration | Initiale Daten zur explorativen Analyse |
| `df_edit` | Preparation & Processing | Bereinigt, transformiert & aggregiert (Arbeitsbasis) |
| `df_final` | Analysis & Reporting | Finaler Output für Insights und Visualisierungen |

> **Best Practice:** Nutze für jeden Transformationsschritt `.copy()`, um die Traceability zu gewährleisten und `df_raw` niemals zu überschreiben.

---

## Setup

In [ ]:
import polars as pl
import pandas as pd
from pathlib import Path

from zh_tram_flow.config import PATHS
from zh_tram_flow.settings import setup_plotting, logger

setup_plotting()
logger.info("00_introduction.ipynb gestartet")

---

## Dateicheck — Master-Datensatz laden

In [ ]:
master_path = PATHS["raw"] / "zh-tram-data-master.parquet"

# Schema prüfen ohne vollständiges Laden
df_schema = pl.read_parquet(master_path, n_rows=1)

print(f"Datei: {master_path}")
print(f"Spalten: {len(df_schema.columns)}")
print()
print(df_schema.schema)

In [ ]:
# Erste Zeilen anschauen
df_sample = pl.read_parquet(master_path, n_rows=5)
df_sample

In [ ]:
# Zeilenanzahl (lazy, ohne alles in RAM zu laden)
row_count = pl.scan_parquet(master_path).select(pl.len()).collect().item()
print(f"Gesamtzeilen: {row_count:,}")

In [ ]:
# GTFS-Referenztabellen prüfen
gtfs_path = PATHS["raw"] / "gtfs"

for f in sorted(gtfs_path.glob("*.parquet")):
    df_tmp = pl.read_parquet(f, n_rows=1)
    print(f"{f.name}: {len(df_tmp.columns)} Spalten")